# S3 · Calibración y ajuste de curvas · De la lectura al valor verdadero
**Sistemas de adquisición de datos · TSU en Electrónica y Pruebas · Jabil**

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ggt13/DAQ_TSU/blob/main/sesion_03_calibracion_y_ajuste/S3_AC_CLASE_Colab_ALUMNO.ipynb)

## Objetivo
Obtener la **curva de calibración** de un sensor a partir de puntos medidos contra una referencia, usarla para
convertir una lectura en el valor físico, y decidir con los **residuos** si basta una recta o hace falta otro modelo.

## La situación
Un transmisor de presión nuevo se compara contra un calibrador patrón en cinco puntos. Su hoja de datos dice
«0.5 V a 4.5 V», pero cada pieza se desvía un poco: por eso se calibra. Cada alumno tiene su propio transmisor
(alcance, ganancia, offset y curvatura). En la práctica se usan mediciones reales de dos potenciómetros hechas por
la generación 2025 con Arduino.

## En qué consiste este cuaderno
| Parte | Tiempo | Qué haces | Qué produce |
|---|---|---|---|
| **A · Actividad en clase** | 30 min | Cuatro ejercicios (★ primero): calibración de dos puntos, conversión de una lectura y no linealidad. En cada uno escribes tu resultado, la sustitución con tus números y su unidad, y lo que predice | La recta de calibración de tu transmisor |
| **B · Práctica** | 80 min | Eliges el rango útil del potenciómetro rotatorio, ajustas la recta, conviertes tu lectura a ángulo y comparas recta contra polinomio en el deslizable | Gráficas firmadas, tu conclusión y el archivo `S3_<código>.csv` |
| **Hoja de resultados** | en la práctica | Importas el CSV en tu hoja de cálculo de Google y contestas las preguntas | El reporte que entregas |

## Qué entregas y cómo se califica
1. **Este cuaderno**, al terminar la sesión, con todo ejecutado. ★ 3 pts · ejercicios 2, 3 y 4, 2 pts cada uno ·
   cierre 1 pt. Se califica que el resultado sea correcto para **tus** datos y que la sustitución y la predicción
   sean tuyas.
2. **La hoja de resultados** con el reporte, en 7 días naturales.

## Reglas
- **Archivo → Guardar una copia en Drive** antes de empezar; trabaja en tu copia.
- Completa sólo lo marcado con ✎. Ninguna celda avanza si falta algo: el mensaje dice qué.
- La revisión automática dice si el ejercicio está **completo**, no si está **bien**; eso lo califica el profesor.
- Si usas un asistente de IA, dilo en tu reporte. Las sustituciones y las respuestas llevan **tus** números.

## Cómo se usa este cuaderno · no necesitas saber programar
Sólo cambias lo que está después del signo **=** en los renglones marcados con ✎.

| Para… | Haz esto |
|---|---|
| Ejecutar una celda | Clic en ▶ a su izquierda (o Ctrl + Enter). Ejecuta las celdas en orden, de arriba hacia abajo |
| Escribir un número | Después del **=**, con punto decimal y sin comas de miles: `u_A = 0.0316`, `MEMORIA = 32000`. Usa al menos 3 cifras significativas |
| Escribir un texto | Entre comillas: `SUST_1 = "70 / 2^10 = 0.0684 °C"`. No borres las comillas |
| Calcular | Con calculadora o con tu hoja de cálculo. Si sabes Python puedes escribir la operación en vez del número: `+ − * /`, potencia `**`, raíz `np.sqrt(x)`, eˣ `np.exp(x)` |
| Ver tus datos | Están impresos debajo de la celda «Tus datos»; usa esos números |
| Un mensaje en rojo | Lee la **última línea**: dice qué falta. Corrige y vuelve a ejecutar esa celda |
| Antes de entregar | **Entorno de ejecución → Reiniciar y ejecutar todo**, y revisa que todo salga ✔ |

In [ ]:
#@title Herramientas del curso (no modificar)
"""Herramientas comunes del curso «Sistemas de adquisición de datos» 2027.

Identificación del alumno, datos personales a partir de su código, firma de las gráficas,
huella de los resultados y CSV firmado para la hoja de resultados.

Este archivo es la fuente única: los cuadernos de cada sesión copian su contenido en la celda
«Herramientas del curso». Si se corrige aquí, se regeneran los cuadernos.
"""
import csv
import datetime as _dt
import hashlib
import zlib

import numpy as np

try:
    from zoneinfo import ZoneInfo
    _ZONA = ZoneInfo("America/Mexico_City")
except Exception:          # Python sin zoneinfo: se usa la hora del sistema
    _ZONA = None

_VALORES_DE_EJEMPLO = {"", "TU NOMBRE", "NOMBRE APELLIDO", "000000000", "123456789"}


class Alumno:
    """Datos de quien ejecuta el cuaderno. Se crea con identificar()."""

    def __init__(self, nombre, codigo, sesion):
        self.nombre = nombre
        self.codigo = codigo
        self.sesion = sesion

    @property
    def semilla(self):
        return int(self.codigo) % (2 ** 32)

    def rng(self, etiqueta=""):
        """Generador de números aleatorios propio del alumno.

        Cada etiqueta da una secuencia independiente: agregar un parámetro nuevo
        no cambia los que ya existían.
        """
        return np.random.default_rng([self.semilla, zlib.crc32(etiqueta.encode("utf-8"))])

    def __repr__(self):
        return f"Alumno({self.nombre!r}, {self.codigo!r}, {self.sesion!r})"


def identificar(nombre, codigo, sesion):
    """Valida la celda de identificación y devuelve el Alumno.

    Detiene el cuaderno con un mensaje claro si el nombre o el código faltan
    o se dejaron con el valor de ejemplo.
    """
    nombre = " ".join(str(nombre).split())
    codigo = str(codigo).strip()
    if nombre.upper() in _VALORES_DE_EJEMPLO or len(nombre.split()) < 2:
        raise ValueError("Escribe tu nombre completo (nombre y apellido) en NOMBRE y vuelve a ejecutar.")
    if codigo in _VALORES_DE_EJEMPLO or not codigo.isdigit() or not 7 <= len(codigo) <= 10:
        raise ValueError("Escribe tu código de alumno (sólo dígitos, de 7 a 10) en CODIGO y vuelve a ejecutar.")
    return Alumno(nombre, codigo, sesion)


def parametro(alumno, etiqueta, minimo, maximo, decimales=3):
    """Un valor personal entre minimo y maximo, siempre el mismo para el mismo código."""
    valor = alumno.rng(etiqueta).uniform(minimo, maximo)
    return round(float(valor), decimales)


def asignar(alumno, opciones, salto=0):
    """Elige una opción de la lista según el código (estación, ventana de fechas, carga del motor...)."""
    opciones = list(opciones)
    return opciones[(int(alumno.codigo) + salto) % len(opciones)]


def _redondea(valor):
    if isinstance(valor, (int, float, np.floating, np.integer)):
        return format(float(valor), ".4g")
    return str(valor)


def huella(alumno, resultados):
    """Seis caracteres que dependen del código, la sesión y los resultados (redondeados a 4 cifras)."""
    partes = [alumno.codigo, alumno.sesion] + [f"{k}={_redondea(v)}" for k, v in sorted(resultados.items())]
    return hashlib.sha256("|".join(partes).encode("utf-8")).hexdigest()[:6].upper()


def ahora():
    momento = _dt.datetime.now(_ZONA) if _ZONA else _dt.datetime.now()
    return momento.strftime("%Y-%m-%d %H:%M")


def titulo(ax, texto, alumno):
    """Título de la gráfica con el nombre del alumno."""
    ax.set_title(f"{texto} · {alumno.nombre}")


def firma(fig, alumno, resultados=None):
    """Pie de la figura: nombre · código · sesión · fecha y hora · huella.

    Toda gráfica que se entrega debe pasar por aquí.
    """
    texto = f"{alumno.nombre} · {alumno.codigo} · {alumno.sesion} · {ahora()}"
    if resultados:
        texto += f" · huella {huella(alumno, resultados)}"
    fig.text(0.01, 0.005, texto, fontsize=8, color="0.35", ha="left", va="bottom")
    fig.subplots_adjust(bottom=max(fig.subplotpars.bottom, 0.14))
    fig._firmada = True
    return fig


def exportar_csv(ruta, alumno, resultados, columnas, extra=None):
    """Escribe el CSV firmado que se importa en la pestaña «CSV» de la hoja de resultados.

    Renglones de metadatos: nombre, codigo, sesion, huella, ejecutado y los de `extra` (p. ej. la fuente
    de los datos). Después, un renglón por resultado («resultado:<nombre>», valor). Luego un renglón vacío
    y la tabla: encabezados y datos. `columnas` es un diccionario {encabezado: lista de valores}, todas de
    la misma longitud. La hoja busca cada dato por su etiqueta, no por su renglón.
    """
    claves = [k.lower() for k in resultados]
    if len(set(claves)) != len(claves):    # la hoja busca sin distinguir mayúsculas: U y u serían el mismo dato
        raise ValueError("Dos resultados tienen el mismo nombre salvo mayúsculas; renómbrelos.")
    largos = {len(v) for v in columnas.values()}
    if len(largos) != 1:
        raise ValueError("Todas las columnas de la tabla deben tener el mismo número de renglones.")
    with open(ruta, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["nombre", alumno.nombre])
        w.writerow(["codigo", alumno.codigo])
        w.writerow(["sesion", alumno.sesion])
        w.writerow(["huella", huella(alumno, resultados)])
        w.writerow(["ejecutado", ahora()])
        for k, v in (extra or {}).items():
            w.writerow([k, v])
        for k, v in resultados.items():
            w.writerow([f"resultado:{k}", _redondea(v)])
        w.writerow([])
        w.writerow(list(columnas))
        for fila in zip(*columnas.values()):
            w.writerow([_redondea(x) for x in fila])
    return ruta


def leer_csv_firmado(ruta):
    """Para el profesor: devuelve (metadatos, resultados, tabla) de un CSV firmado."""
    meta, resultados, tabla = {}, {}, []
    with open(ruta, newline="", encoding="utf-8") as f:
        filas = list(csv.reader(f))
    i = 0
    while i < len(filas) and filas[i]:
        clave, valor = filas[i][0], filas[i][1]
        if clave.startswith("resultado:"):
            resultados[clave.split(":", 1)[1]] = float(valor)
        else:
            meta[clave] = valor
        i += 1
    tabla = filas[i + 1:] if i + 1 < len(filas) else []
    return meta, resultados, tabla


def verificar_csv(ruta):
    """Para el profesor: True si la huella del CSV corresponde a su código, sesión y resultados."""
    meta, resultados, _ = leer_csv_firmado(ruta)
    alumno = Alumno(meta["nombre"], meta["codigo"], meta["sesion"])
    return huella(alumno, resultados) == meta["huella"]


# ---------------------------------------------------------------------------------------------
# Revisión de lo que escribe el alumno. Comprueba que cada ejercicio esté COMPLETO, no que esté
# bien: el resultado correcto lo califica el profesor con la clave. Si falta algo, detiene el
# cuaderno y dice qué falta.
# ---------------------------------------------------------------------------------------------
import math
import re

REGISTRO = {}


def _normaliza(texto):
    t = str(texto).replace("\u202f", "").replace("\u00a0", "").replace(" ", "")
    t = re.sub(r"(?<![\d.])([1-9]\d{0,2})((?:,\d{3})+)(?![\d])",  # separador de miles: 32,000 -> 32000
               lambda m: m.group(1) + m.group(2).replace(",", ""), t)
    return t.replace(",", ".")                                       # coma decimal: 0,068 -> 0.068


def _formas(x):
    """Maneras razonables de escribir un número: entero, o redondeado a 2, 3 o 4 cifras significativas."""
    x = float(x)
    formas = set()
    if x == int(x) and abs(x) < 1e9:
        formas.add(str(int(x)))
    if abs(x) >= 100:
        formas.add(str(int(round(x))))                       # 34271.6 escrito como 34272
    if abs(x) >= 1:
        for dec in (1, 2):
            formas.add(f"{x:.{dec}f}".rstrip("0").rstrip("."))  # 123456.7 con sus decimales
    for cifras in (2, 3, 4, 5, 6):
        g = f"{x:.{cifras}g}"
        if "e" not in g:
            formas.add(g)
    return {f for f in formas if f not in ("", "-0")}


def aparece(x, texto):
    """True si el número x está escrito en el texto (sin confundir 0.03 con 0.0337)."""
    t = _normaliza(texto)
    def patron(f):
        cola = r"0*" if "." in f else r"(?:\.0*)?"            # 0.0342 = 0.034200 · 120 = 120.0, pero no 1200
        return r"(?<![\d.])" + re.escape(f) + cola + r"(?![\d])"
    return any(re.search(patron(f), t) for f in _formas(x))


def _palabras(texto):
    return len(re.findall(r"\w+", str(texto)))


def revisa(n, valor, sustitucion, prediccion, datos, unidades):
    """Revisión de un ejercicio de la parte A.

    valor: lo que calculó la expresión del alumno. sustitucion: texto con sus números, el resultado y la
    unidad (p. ej. «70 / 2**8 = 0.273 °C»). prediccion: qué espera ver en la práctica, con sus palabras.
    datos: los números personales que deben aparecer en la sustitución. unidades: unidades aceptadas.
    """
    faltan = []
    if valor is None:
        faltan.append("la expresión en Python")
    elif not isinstance(valor, (int, float, np.integer, np.floating)) or not math.isfinite(float(valor)):
        faltan.append("una expresión que dé un número")
    s = str(sustitucion)
    if "=" not in s:
        faltan.append("la sustitución con «=»")
    # datos: lista de números (se nombran por su valor) o {descripción: valor} (se nombran sin revelar el valor)
    pares = datos.items() if isinstance(datos, dict) else [(None, d) for d in datos]
    for que, d in pares:
        if not aparece(d, s):
            faltan.append(f"{que} en la sustitución" if que else f"tu dato {float(d):g} en la sustitución")
    if valor is not None and not faltan and not aparece(valor, s.split("=")[-1]):
        faltan.append("el resultado que dio tu expresión, después del último «=»")
    if not any(u in s for u in unidades):
        faltan.append("la unidad (" + " o ".join(unidades) + ")")
    if _palabras(prediccion) < 10:
        faltan.append("lo que predice, en al menos 10 palabras")
    REGISTRO[n] = not faltan
    if faltan:
        raise ValueError(f"Ejercicio {n}: falta " + "; ".join(faltan) + ".")
    print(f"✔ Ejercicio {n} completo.")


def revisa_texto(nombre, texto, numeros, minimo=20):
    """Revisión de una respuesta escrita: largo mínimo y los números propios que debe citar.

    numeros: diccionario {qué es: valor}; el mensaje nombra lo que falta sin revelar el valor.
    """
    faltan = [que for que, x in numeros.items() if not aparece(x, texto)]
    if _palabras(texto) < minimo:
        faltan.insert(0, f"al menos {minimo} palabras")
    REGISTRO[nombre] = not faltan
    if faltan:
        raise ValueError(f"{nombre}: falta " + "; ".join(faltan) + ".")
    print(f"✔ {nombre} completo.")


import matplotlib.pyplot as plt
plt.rcParams.update({'figure.figsize': (9, 3.6), 'axes.spines.top': False, 'axes.spines.right': False})

def completa(valor, que):
    if valor is None:
        raise ValueError(f'Completa {que} (✎) y vuelve a ejecutar.')
    return valor

# Mediciones reales de la generación 2025 (Arduino, 10 bits, 5 V)
GRADOS = np.array([0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120, 130, 140, 150, 160, 170, 180, 190, 200, 210,
                   220, 230, 240, 250, 260, 270, 280, 290])
SUBIDA = np.array([0.0, 0.0, 0.005, 0.22, 0.415, 0.606, 0.821, 1.007, 1.183, 1.393, 1.603, 1.808, 2.028, 2.278, 2.473,
                   2.669, 2.874, 3.079, 3.309, 3.524, 3.739, 3.93, 4.12, 4.301, 4.506, 4.663, 4.848, 5.0, 5.0, 5.0])
BAJADA = np.array([0, 0, 0.059, 0.244, 0.45, 0.65, 0.86, 1.017, 1.217, 1.403, 1.65, 1.818, 2.038, 2.258, 2.463, 2.654,
                   2.903, 3.113, 3.338, 3.529, 3.719, 3.935, 4.125, 4.326, 4.497, 4.687, 4.795, 5, 5, 5])
CM = np.array([0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0])
V_DESL = np.array([0.0, 0.044, 0.225, 0.538, 1.408, 2.278, 2.947, 3.456, 3.978, 4.492, 4.814, 4.951, 5.0])
FUENTE = "Mediciones de la generación 2025 (Arduino, 10 bits, 5 V)"


def datos_s3(alumno):
    """Transmisor de presión con 0.5 V a 4.5 V nominales; puntos de calibración a 10, 30, 50, 70 y 90 % del alcance."""
    alcance = asignar(alumno, [6, 10, 16, 25, 40])
    g = parametro(alumno, "ganancia", 0.95, 1.05, 4)
    off = parametro(alumno, "offset", -0.08, 0.08, 3)
    curva = parametro(alumno, "curva", -0.06, 0.06, 3)
    x = np.round(alcance * np.array([0.1, 0.3, 0.5, 0.7, 0.9]), 2)
    f = x / alcance
    y = 0.5 + off + 4.0 * g * f + curva * 4 * f * (1 - f) + alumno.rng("ruido").normal(0, 0.004, x.size)
    v_nueva = parametro(alumno, "v_nueva", 1.2, 4.0, 3)
    return dict(ALCANCE=alcance, X_REF=x, Y_LECT=np.round(y, 3), V_NUEVA=v_nueva,
                V_LECT_B=parametro(alumno, "v_lect_b", 0.6, 4.3, 3))


def clave_s3(alumno):
    d = datos_s3(alumno)
    x, y = d["X_REF"], d["Y_LECT"]
    m = (y[-1] - y[0]) / (x[-1] - x[0])
    b = y[0] - m * x[0]
    return {"m_2p": m, "b_2p": b, "P_corr": (d["V_NUEVA"] - b) / m, "no_lin_V": float(abs(y[2] - (m * x[2] + b)))}


def ajuste(x, y, grado):
    c = np.polyfit(x, y, grado)
    yh = np.polyval(c, x)
    r2 = 1 - np.sum((y - yh) ** 2) / np.sum((y - np.mean(y)) ** 2)
    return c, yh, float(r2)


def revisa_rango(desde, hasta):
    """El rango útil no debe incluir puntos saturados (salida pegada a 0 V o a 5 V)."""
    if desde is None or hasta is None:
        raise ValueError("Escribe DESDE y HASTA, en grados.")
    k = (GRADOS >= desde) & (GRADOS <= hasta)
    sat = GRADOS[k & ((SUBIDA < 0.05) | (SUBIDA > 4.95))]
    faltan = []
    if k.sum() < 8:
        faltan.append("deja al menos 8 puntos para el ajuste")
    if len(sat):
        faltan.append("incluiste puntos saturados en " + ", ".join(f"{g}°" for g in sat))
    REGISTRO["Rango útil"] = not faltan
    if faltan:
        raise ValueError("Rango útil: " + "; ".join(faltan) + ".")
    print(f"✔ Rango útil de {desde}° a {hasta}°: {k.sum()} puntos, ninguno saturado.")

print('Herramientas del curso listas.')

## Identificación
Tu nombre y tu código aparecen en cada gráfica y en el CSV. Sin ellos el cuaderno no avanza.

In [ ]:
NOMBRE = ""   # ✎ tu nombre completo
CODIGO = ""   # ✎ tu código de alumno
SESION = "S3"

alumno = identificar(NOMBRE, CODIGO, SESION)
print("Cuaderno de", alumno.nombre, "· código", alumno.codigo)

## Parte A · Actividad en clase (30 min)
Ejecuta la celda para ver **tu** transmisor: los cinco puntos de calibración (`X_REF`, presión del patrón en bar;
`Y_LECT`, salida del transmisor en V) y una lectura nueva `V_NUEVA` que hay que convertir. El primer punto es el
primer número de cada lista; el último, el último.

**Cómo se llena cada ejercicio.** Ejemplo con otros datos (no son los tuyos):
```
m_2p   = 0.2025
SUST_1 = "(4.12 − 0.88) / (18 − 2) = 0.2025 V/bar"
PRED_1 = "cada bar más de presión sube la salida unas dos décimas de volt en todo el alcance"
```

In [ ]:
globals().update(datos_s3(alumno))
print(f"Tu transmisor · alcance 0 a {ALCANCE} bar · lectura nueva V_NUEVA = {V_NUEVA} V")
print("  X_REF (bar):", ", ".join(f"{x:g}" for x in X_REF))
print("  Y_LECT (V): ", ", ".join(f"{y:.3f}" for y in Y_LECT))

### ★ Ejercicio 1 · Pendiente de la calibración de dos puntos (3 pts)
**Para qué sirve:** es la sensibilidad real de **tu** pieza, no la del catálogo. Con ella se convierte cualquier lectura.

Con el primer y el último punto: **m = (y₂ − y₁) / (x₂ − x₁)**. Calcula `m_2p` en V/bar.

In [ ]:
# ✎ 1a · Tu resultado: el número que calculaste (punto decimal, al menos 3 cifras significativas)
m_2p = None   # ✎ escribe aquí
# ✎ 1b · La sustitución con TUS números, el resultado y la unidad
SUST_1 = ""   # ✎ escribe aquí
# ✎ 1c · Lo que predice en la práctica, con tus palabras (al menos 10)
PRED_1 = ""   # ✎ escribe aquí

revisa(1, m_2p, SUST_1, PRED_1, datos={'tu primer voltaje': Y_LECT[0], 'tu último voltaje': Y_LECT[-1], 'tu primera presión': X_REF[0], 'tu última presión': X_REF[-1]}, unidades=['V/bar'])

### Ejercicio 2 · Ordenada al origen (2 pts)
**Para qué sirve:** es lo que saldría sin presión: el offset real de tu pieza.

La recta pasa por tu primer punto; despeja **b** de y = m·x + b. Calcula `b_2p` en V.

In [ ]:
# ✎ 2a · Tu resultado: el número que calculaste (punto decimal, al menos 3 cifras significativas)
b_2p = None   # ✎ escribe aquí
# ✎ 2b · La sustitución con TUS números, el resultado y la unidad
SUST_2 = ""   # ✎ escribe aquí
# ✎ 2c · Lo que predice en la práctica, con tus palabras (al menos 10)
PRED_2 = ""   # ✎ escribe aquí

revisa(2, b_2p, SUST_2, PRED_2, datos={'tu primer voltaje': Y_LECT[0], 'tu m': m_2p}, unidades=['V'])

### Ejercicio 3 · De la lectura a la presión (2 pts)
**Para qué sirve:** es para lo que se calibra: convertir lo que marca el instrumento en el valor físico.

Despeja **x** de la recta para tu `V_NUEVA`. Calcula `P_corr` en bar.

In [ ]:
# ✎ 3a · Tu resultado: el número que calculaste (punto decimal, al menos 3 cifras significativas)
P_corr = None   # ✎ escribe aquí
# ✎ 3b · La sustitución con TUS números, el resultado y la unidad
SUST_3 = ""   # ✎ escribe aquí
# ✎ 3c · Lo que predice en la práctica, con tus palabras (al menos 10)
PRED_3 = ""   # ✎ escribe aquí

revisa(3, P_corr, SUST_3, PRED_3, datos={'tu dato V_NUEVA': V_NUEVA}, unidades=['bar'])

### Ejercicio 4 · ¿Qué tan recta es? (2 pts)
**Para qué sirve:** si los puntos intermedios se separan mucho de la recta, una calibración de dos puntos no basta.

En el **punto central** (el tercero, 50 % del alcance), calcula qué voltaje da tu recta (m·x + b) y réstalo del
voltaje medido ahí. La **no linealidad** es esa diferencia sin signo. Resultado en V.

In [ ]:
# ✎ 4a · Tu resultado: el número que calculaste (punto decimal, al menos 3 cifras significativas)
no_lin_V = None   # ✎ escribe aquí
# ✎ 4b · La sustitución con TUS números, el resultado y la unidad
SUST_4 = ""   # ✎ escribe aquí
# ✎ 4c · Lo que predice en la práctica, con tus palabras (al menos 10)
PRED_4 = ""   # ✎ escribe aquí

revisa(4, no_lin_V, SUST_4, PRED_4, datos={'tu voltaje central': Y_LECT[2], 'tu presión central': X_REF[2]}, unidades=['V'])

In [ ]:
m_ls, b_ls = np.polyfit(X_REF, Y_LECT, 1)
xx = np.linspace(0, ALCANCE, 100)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.6))
a1.plot(X_REF, Y_LECT, "o", label="puntos de calibración")
a1.plot(xx, m_2p * xx + b_2p, label="recta de dos puntos")
a1.plot(xx, m_ls * xx + b_ls, "--", label="mínimos cuadrados")
a1.set_xlabel("presión del patrón (bar)"); a1.set_ylabel("salida (V)"); a1.legend(fontsize=8)
titulo(a1, "Calibración", alumno)
a2.axhline(0, color="#444", lw=1); a2.vlines(X_REF, 0, Y_LECT - (m_2p * X_REF + b_2p), color="#C62828", lw=3)
a2.set_xlabel("presión (bar)"); a2.set_ylabel("residuo (V)"); a2.set_title("Residuos de la recta de dos puntos")
firma(fig, alumno); plt.tight_layout(rect=(0, 0.06, 1, 1)); plt.show()
print(f"Mínimos cuadrados con los cinco puntos: m_ls = {m_ls:.5f} V/bar · b_ls = {b_ls:.4f} V")

### Cierre (1 pt)
Compara tu pendiente de dos puntos con la de mínimos cuadrados (`m_ls`, impresa arriba). ¿Por qué no son iguales y
cuál usarías? Al menos 20 palabras, con las dos pendientes.

In [ ]:
CIERRE = ""   # ✎ escribe aquí

revisa_texto('Cierre', CIERRE, {'tu pendiente de dos puntos': m_2p,
             'la pendiente de mínimos cuadrados': m_ls})

## Parte B · Práctica (80 min)
**Objetivo:** calibrar con datos reales, elegir el rango útil y el modelo, y medir lo que se gana contra la regla de tres.

Mediciones de la generación 2025 con Arduino: un potenciómetro **rotatorio** medido de 0° a 290° en subida y bajada, y
uno **deslizable** de 0 a 6 cm.

| Bloque | min | Qué haces |
|---|---|---|
| B1 · Rango útil | 20 | Eliges desde qué ángulo y hasta cuál la salida no está saturada |
| B2 · Tu lectura | 20 | Ajustas la recta en tu rango y conviertes tu lectura a ángulo |
| B3 · ¿Recta o polinomio? | 20 | En el deslizable eliges el grado del polinomio mirando los residuos |
| B4 · Conclusión | 10 | Comparas calibración contra regla de tres, con tus números |
| Tabla y captura | 10 | Descargas el CSV y lo importas en tu hoja de resultados |

In [ ]:
fig, ax = plt.subplots()
ax.plot(GRADOS, SUBIDA, "o-", ms=4, label="subida"); ax.plot(GRADOS, BAJADA, "s", ms=3, alpha=0.6, label="bajada")
ax.set_xlabel("ángulo (°)"); ax.set_ylabel("salida (V)"); ax.set_xticks(range(0, 291, 30)); ax.grid(alpha=0.3); ax.legend()
titulo(ax, "Potenciómetro rotatorio, generación 2025", alumno)
firma(fig, alumno); plt.show()
print(f"Tu lectura para convertir: V_LECT_B = {V_LECT_B} V")

### B1 · Rango útil
Mira la gráfica: en los extremos la salida se queda pegada en 0 V o en 5 V (**saturación**). Escribe en `DESDE` y
`HASTA` el primer y el último ángulo (múltiplos de 10) donde la salida **sí** cambia. La celda revisa que no
incluyas puntos saturados.

In [ ]:
DESDE = None   # ✎ ángulo inicial, en grados
HASTA = None   # ✎ ángulo final, en grados

revisa_rango(DESDE, HASTA)
k = (GRADOS >= DESDE) & (GRADOS <= HASTA)
(m_rot, b_rot), yh, R2_rot = ajuste(GRADOS[k], SUBIDA[k], 1)
histeresis_V = float(np.max(np.abs(SUBIDA[k] - BAJADA[k])))
print(f"Recta en {DESDE}°–{HASTA}°: m = {m_rot:.5f} V/° · b = {b_rot:.4f} V · R² = {R2_rot:.5f}")
print(f"Histéresis máxima (subida contra bajada) en tu rango: {histeresis_V:.3f} V")

### B2 · Tu lectura en grados
Con la recta de tu rango (`m_rot`, `b_rot`), convierte tu `V_LECT_B` a ángulo: `ang_cal`. El cuaderno la compara con
la **regla de tres** que usaría alguien sin calibrar (5 V ↔ 290°).

In [ ]:
ang_cal = None   # ✎ escribe aquí

completa(ang_cal, "el ángulo de tu lectura")
ang_regla = V_LECT_B / 5 * 290
xx = np.linspace(DESDE, HASTA, 100)
fig, ax = plt.subplots()
ax.plot(GRADOS, SUBIDA, "o", color="#BBB", label="fuera de tu rango")
ax.plot(GRADOS[k], SUBIDA[k], "o", label="tu rango")
ax.plot(xx, m_rot * xx + b_rot, lw=2, label=f"recta, R² = {R2_rot:.4f}")
ax.axhline(V_LECT_B, ls=":", c="#444"); ax.plot(ang_cal, V_LECT_B, "*", ms=14, c="#D9822B", label=f"tu lectura: {ang_cal:.1f}°")
ax.set_xlabel("ángulo (°)"); ax.set_ylabel("salida (V)"); ax.legend(fontsize=8)
titulo(ax, "Calibración del rotatorio", alumno)
firma(fig, alumno); plt.show()
print(f"Calibración: {ang_cal:.1f}° · regla de tres: {ang_regla:.1f}° · diferencia {ang_regla - ang_cal:+.1f}°")

### B3 · ¿Recta o polinomio? (deslizable)
Prueba en `GRADO_DESL` los grados 1, 2, 3 y 4; ejecuta cada vez y mira **la forma de los residuos**, no sólo R². Deja
el grado que elegirías y explica por qué en tu conclusión.

In [ ]:
GRADO_DESL = None   # ✎ grado del polinomio, de 1 a 4

completa(GRADO_DESL, "el grado del polinomio")
c_d, yh_d, R2_desl = ajuste(CM, V_DESL, int(GRADO_DESL))
RESULTADOS = {"m_2p": m_2p, "b_2p": b_2p, "P_corr": P_corr, "no_lin_V": no_lin_V, "desde": DESDE, "hasta": HASTA,
              "m_rot": m_rot, "b_rot": b_rot, "R2_rot": R2_rot, "v_lect_b": V_LECT_B, "ang_cal": float(ang_cal),
              "ang_regla": ang_regla, "grado_desl": int(GRADO_DESL), "R2_desl": R2_desl}
xx = np.linspace(0, 6, 200)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.6))
a1.plot(CM, V_DESL, "o"); a1.plot(xx, np.polyval(c_d, xx), lw=2)
a1.set_xlabel("posición (cm)"); a1.set_ylabel("salida (V)"); titulo(a1, f"Deslizable, grado {int(GRADO_DESL)}", alumno)
a2.axhline(0, color="#444", lw=1); a2.vlines(CM, 0, V_DESL - yh_d, color="#C62828", lw=3)
a2.set_xlabel("posición (cm)"); a2.set_ylabel("residuo (V)"); a2.set_title(f"R² = {R2_desl:.4f}")
firma(fig, alumno, RESULTADOS); plt.tight_layout(rect=(0, 0.06, 1, 1)); plt.show()

### B4 · Conclusión
¿Cuánto cambia tu lectura con la calibración frente a la regla de tres, y por qué? ¿Qué grado elegiste para el
deslizable? Al menos 25 palabras, con tus dos ángulos.

In [ ]:
CONCLUSION_B = ""   # ✎ escribe aquí

revisa_texto('Conclusión de la práctica', CONCLUSION_B,
             {'tu ángulo calibrado': ang_cal, 'tu ángulo por regla de tres': ang_regla}, minimo=25)

### Tabla y captura · CSV para tu hoja de resultados
La celda descarga `S3_<código>.csv`. En tu hoja de resultados: abre la pestaña **CSV** → **Archivo → Importar →
Subir** → elige el archivo → **Reemplazar la hoja actual**. La pestaña Resultados se llena sola.

In [ ]:
recta = np.where(k, m_rot * GRADOS + b_rot, np.nan)
ruta = f"S3_{alumno.codigo}.csv"
exportar_csv(ruta, alumno, RESULTADOS,
             {"grados": GRADOS, "V_subida": SUBIDA, "V_recta": ["" if np.isnan(v) else v for v in recta]},
             extra={"fuente": FUENTE, "rango": f"{DESDE} a {HASTA} grados"})
print("Archivo listo:", ruta, "· huella", huella(alumno, RESULTADOS))
try:
    from google.colab import files
    files.download(ruta)
except ImportError:
    pass

## Cierre del cuaderno
Antes de entregar: **Entorno de ejecución → Reiniciar y ejecutar todo**. Todo debe salir ✔.

In [ ]:
for k, ok in REGISTRO.items():
    print(("✔" if ok else "✘"), k if isinstance(k, str) else f"Ejercicio {k}")
print(f"\n{alumno.nombre} · {alumno.codigo} · {SESION} · {ahora()} · fuente: {FUENTE} · huella {huella(alumno, RESULTADOS)}")